In [1]:
import ee

ee.Initialize(project="ee-wildfire-recovery-vasant")

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [2]:
# contains completed MTBS fire polygons and metadata, with coverage from 1984 through 2024
mtbs = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

print("MTBS dataset loaded.")

MTBS dataset loaded.


# understanding schema

In [3]:
print("How many fire-event polygons are in this entire collection?:", mtbs.size().getInfo())

first_fire = mtbs.first()

print("\nAvailable properties:")
print(first_fire.propertyNames().getInfo())

How many fire-event polygons are in this entire collection?: 30730

Available properties:
['Comment', 'Incid_Type', 'High_T', 'NoData_T', 'Mod_T', 'BurnBndLon', 'Ig_Date', 'Low_T', 'Map_Prog', 'dNBR_stdDv', 'Event_ID', 'Incid_Name', 'Map_ID', 'Pre_ID', 'Perim_ID', 'BurnBndLat', 'BurnBndAc', 'dNBR_offst', 'IncGreen_T', 'irwinID', 'Post_ID', 'Asmnt_Type', 'system:index']


In [4]:
# 30,730 = total completed MTBS burned-area event polygons in the collection
# Event_ID = unique event identifier
# Incid_Name = incident/fire name
# Ig_Date = ignition date
# Incid_Type = event type, such as wildfire vs prescribed fire
# BurnBndAc = mapped burned-boundary acreage
# BurnBndLat / BurnBndLon = location coordinates
# Low_T, Mod_T, High_T = analyst-selected severity thresholds
# dNBR_offst = dNBR offset used in the MTBS mapping process
# Pre_ID / Post_ID = IDs for the pre- and post-fire imagery used in mapping
# Perim_ID / Map_ID = MTBS mapping/perimeter identifiers

# arbitrary first record

In [5]:
first_fire_info = first_fire.toDictionary().getInfo()

first_fire_info

{'Asmnt_Type': 'Extended',
 'BurnBndAc': 69376,
 'BurnBndLat': '65.717',
 'BurnBndLon': '-145.566',
 'Comment': 'followed state perimeter between Preacher and Central Complex 2004',
 'Event_ID': 'AK6571714556620050612',
 'High_T': 325,
 'Ig_Date': 1118559600000,
 'IncGreen_T': -250,
 'Incid_Name': 'PREACHER CREEK',
 'Incid_Type': 'Wildfire',
 'Low_T': -50,
 'Map_ID': 13972,
 'Map_Prog': 'MTBS',
 'Mod_T': 127,
 'NoData_T': -1070,
 'Perim_ID': '',
 'Post_ID': '706801420060904',
 'Pre_ID': '506801420040906',
 'dNBR_offst': -115,
 'dNBR_stdDv': -9999,
 'irwinID': ''}

In [6]:
# Fire: PREACHER CREEK
# Type: Wildfire
# Area: 69,376 acres
# Location: roughly 65.7° N, 145.6° W — Alaska, not in the 11 states picked
# Event_ID contains the event's identifying information
# Ig_Date is stored as a numeric timestamp rather than a human-readable date
# Low_T, Mod_T, High_T are MTBS's severity thresholds for this event
# Pre_ID / Post_ID identify imagery used in MTBS's pre/post-fire assessment
# dNBR_offst is part of MTBS's dNBR calibration/mapping information
# Values like -9999 are a sentinel/missing value, will need to recognize later rather than accidentally treating as a real measurement.

# Arbitrary first record to understand the schema. (this fire is not usable in the project)

# filtering toward actual study population

### Wildfire only
### Ignition dates from 2013 through 2020

In [7]:
wildfires_2013_2020 = (
    mtbs
    .filter(ee.Filter.eq("Incid_Type", "Wildfire"))
    .filter(
        ee.Filter.gte(
            "Ig_Date",
            ee.Date("2013-01-01").millis()
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            ee.Date("2021-01-01").millis()
        )
    )
)

print(
    "MTBS wildfires from 2013-2020:",
    wildfires_2013_2020.size().getInfo()
)

MTBS wildfires from 2013-2020: 3644


### Building 11-state western region (not filtering fires to these regions yet, just creating a geographical object representing the 11 states)

In [8]:
states = ee.FeatureCollection("TIGER/2018/States")

western_state_codes = [
    "WA", "OR", "CA", "ID", "NV", "MT",
    "WY", "UT", "CO", "AZ", "NM"
]

western_states = states.filter(
    ee.Filter.inList("STUSPS", western_state_codes)
)

print("Western states selected:", western_states.size().getInfo())

Western states selected: 11


### Only keeping wildfires that intersect with these states

In [9]:
western_wildfires_2013_2020 = wildfires_2013_2020.filterBounds(
    western_states.geometry()
)

print(
    "Western U.S. MTBS wildfires from 2013-2020:",
    western_wildfires_2013_2020.size().getInfo()
)

Western U.S. MTBS wildfires from 2013-2020: 2282


### Selecting sensible Proof-of-concept candidate pool that makes it easier to test other datasets with

### Note this isnt actually filtering down eligible fires, it is just selecting ones which make sense to be candidates for testing our first test case of combining multiple datasets

In [10]:
# 2017–2019 (comfortable pre-fire years and Season 5 data while staying largely in the Landsat 8 era.)
# 10,000–100,000 acres (thousands of potential 90 m patches and ecological variation, but avoids beginning with either tiny fires or enormous fire complexes)
# already restricted to our 11 western states
# wildfire only

poc_candidates = (
    western_wildfires_2013_2020
    .filter(
        ee.Filter.gte(
            "Ig_Date",
            ee.Date("2017-01-01").millis()
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            ee.Date("2020-01-01").millis()
        )
    )
    .filter(ee.Filter.gte("BurnBndAc", 10000))
    .filter(ee.Filter.lte("BurnBndAc", 100000))
)

print(
    "Initial POC candidate fires:",
    poc_candidates.size().getInfo()
)



Initial POC candidate fires: 243


In [11]:
## many candidates remain so inspecting the biggest 20 first
candidate_sample = (
    poc_candidates
    .sort("BurnBndAc", False)
    .limit(20)
)

names = candidate_sample.aggregate_array("Incid_Name").getInfo()
acres = candidate_sample.aggregate_array("BurnBndAc").getInfo()
dates_ms = candidate_sample.aggregate_array("Ig_Date").getInfo()
lats = candidate_sample.aggregate_array("BurnBndLat").getInfo()
lons = candidate_sample.aggregate_array("BurnBndLon").getInfo()
event_ids = candidate_sample.aggregate_array("Event_ID").getInfo()

from datetime import datetime, timezone

for name, acre, date_ms, lat, lon, event_id in zip(
    names, acres, dates_ms, lats, lons, event_ids
):
    date = datetime.fromtimestamp(
        date_ms / 1000, tz=timezone.utc
    ).strftime("%Y-%m-%d")

    print(
        f"{date} | "
        f"{name:30} | "
        f"{acre:>8,.0f} acres | "
        f"({lat}, {lon}) | "
        f"{event_id}"
    )

2018-06-21 | BOXCAR 0410 RN                 |   99,874 acres | (45.022, -121.004) | OR4516012107120180621
2017-07-03 | TRUCKEE                        |   98,411 acres | (39.772, -119.194) | NV3976211925220170703
2018-11-08 | WOOLSEY                        |   97,962 acres | (34.125, -118.824) | CA3424011870020181108
2018-07-13 | FERGUSON                       |   97,307 acres | (37.635, -119.807) | CA3765211988120180713
2017-08-12 | OAK                            |   95,692 acres | (41.747, -123.58) | CA4178612347520170812
2017-07-29 | HIGHLINE                       |   93,699 acres | (45.364, -115.156) | ID4549011522720170729
2017-08-29 | TOHAKUM 2                      |   92,711 acres | (40.276, -119.374) | NV4027311946420170830
2018-06-30 | COUNTY                         |   92,450 acres | (38.683, -122.155) | CA3881012220020180630
2017-08-30 | SARTIN DRAW                    |   91,570 acres | (45.855, -106.027) | MT4573910600920170830
2017-08-31 | DEER CREEK                     |  

In [12]:
# inspecting FERGUSON
ferguson_id = "CA3765211988120180713"

ferguson = ee.Feature(
    poc_candidates
    .filter(ee.Filter.eq("Event_ID", ferguson_id))
    .first()
)

ferguson_date = ee.Date(ferguson.get("Ig_Date"))
ferguson_geom = ferguson.geometry()

# POC-only prior-fire screen: 7 years before the focal fire
prior_start = ferguson_date.advance(-7, "year")

prior_fires = (
    mtbs
    .filter(ee.Filter.eq("Incid_Type", "Wildfire"))
    .filter(
        ee.Filter.gte(
            "Ig_Date",
            prior_start.millis()
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            ferguson_date.millis()
        )
    )
    .filterBounds(ferguson_geom)
)

# Ferguson burned in 2018:
# Season 1 = 2019, Season 5 = 2023
season5_end = ee.Date("2023-10-01")

reburn_fires = (
    mtbs
    .filter(ee.Filter.eq("Incid_Type", "Wildfire"))
    .filter(
        ee.Filter.gt(
            "Ig_Date",
            ferguson_date.millis()
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            season5_end.millis()
        )
    )
    .filterBounds(ferguson_geom)
)

print("Prior overlapping fires:", prior_fires.size().getInfo())
print("Prior fire names:", prior_fires.aggregate_array("Incid_Name").getInfo())

print("\nReburns through Season 5:", reburn_fires.size().getInfo())
print("Reburn names:", reburn_fires.aggregate_array("Incid_Name").getInfo())

Prior overlapping fires: 4
Prior fire names: ['EL PORTAL', 'RIM', 'AVALANCHE', 'MOTOR']

Reburns through Season 5: 2
Reburn names: ['BRICEBURG', 'OAK']


In [13]:
# measuring overlap
ACRES_PER_SQ_METER = 1 / 4046.8564224

ferguson_area_acres = ferguson_geom.area(maxError=30).multiply(
    ACRES_PER_SQ_METER
)

def add_overlap_stats(feature):
    overlap_geom = feature.geometry().intersection(
        ferguson_geom,
        maxError=30
    )

    overlap_acres = overlap_geom.area(maxError=30).multiply(
        ACRES_PER_SQ_METER
    )

    overlap_pct = overlap_acres.divide(
        ferguson_area_acres
    ).multiply(100)

    return feature.set({
        "overlap_acres": overlap_acres,
        "overlap_pct": overlap_pct
    })


prior_with_overlap = prior_fires.map(add_overlap_stats)
reburn_with_overlap = reburn_fires.map(add_overlap_stats)


for label, collection in [
    ("PRIOR FIRES", prior_with_overlap),
    ("REBURNS", reburn_with_overlap)
]:
    print(f"\n{label}")

    info = collection.getInfo()

    for feature in info["features"]:
        props = feature["properties"]

        print(
            f"{props['Incid_Name']:20} | "
            f"{props['overlap_acres']:>10,.0f} overlap acres | "
            f"{props['overlap_pct']:>6.2f}% of Ferguson"
        )


PRIOR FIRES
EL PORTAL            |        916 overlap acres |   0.94% of Ferguson
RIM                  |        365 overlap acres |   0.38% of Ferguson
AVALANCHE            |        595 overlap acres |   0.61% of Ferguson
MOTOR                |      2,011 overlap acres |   2.07% of Ferguson

REBURNS
BRICEBURG            |          3 overlap acres |   0.00% of Ferguson
OAK                  |      4,256 overlap acres |   4.38% of Ferguson


In [14]:
# comparing 6 other candidates at the same time by checking overlap with prev fires and reburns in the same windows

import pandas as pd

ACRES_PER_SQ_METER = 1 / 4046.8564224


def screen_fire_history(event_id, prior_lookback_years=7):
    # Get focal fire
    focal = ee.Feature(
        western_wildfires_2013_2020
        .filter(ee.Filter.eq("Event_ID", event_id))
        .first()
    )

    focal_date = ee.Date(focal.get("Ig_Date"))
    focal_year = ee.Number(focal_date.get("year"))
    focal_geom = focal.geometry()

    focal_area_m2 = focal_geom.area(maxError=30)

    # -------------------------
    # PRIOR-FIRE SCREEN
    # -------------------------
    prior_start = focal_date.advance(-prior_lookback_years, "year")

    prior_fires = (
        mtbs
        .filter(ee.Filter.eq("Incid_Type", "Wildfire"))
        .filter(ee.Filter.gte("Ig_Date", prior_start.millis()))
        .filter(ee.Filter.lt("Ig_Date", focal_date.millis()))
        .filterBounds(focal_geom)
    )

    # Union all overlapping prior-fire geometries first,
    # then measure UNIQUE overlap with focal fire.
    prior_union = prior_fires.geometry(maxError=30)

    prior_overlap_m2 = (
        prior_union
        .intersection(focal_geom, maxError=30)
        .area(maxError=30)
    )

    prior_overlap_pct = (
        prior_overlap_m2
        .divide(focal_area_m2)
        .multiply(100)
    )

    # -------------------------
    # REBURN SCREEN
    # -------------------------
    # For these summer 2017-2019 POC candidates,
    # Season 1 is the following growing season,
    # so Season 5 ends Sept. 30 of focal_year + 5.
    season5_end = ee.Date.fromYMD(
        focal_year.add(5),
        10,
        1
    )

    reburn_fires = (
        mtbs
        .filter(ee.Filter.eq("Incid_Type", "Wildfire"))
        .filter(ee.Filter.gt("Ig_Date", focal_date.millis()))
        .filter(ee.Filter.lt("Ig_Date", season5_end.millis()))
        .filterBounds(focal_geom)
    )

    reburn_union = reburn_fires.geometry(maxError=30)

    reburn_overlap_m2 = (
        reburn_union
        .intersection(focal_geom, maxError=30)
        .area(maxError=30)
    )

    reburn_overlap_pct = (
        reburn_overlap_m2
        .divide(focal_area_m2)
        .multiply(100)
    )

    # small summary
    return {
        "event_id": event_id,
        "name": focal.get("Incid_Name").getInfo(),
        "year": int(focal_year.getInfo()),
        "acres": focal.get("BurnBndAc").getInfo(),
        "prior_fire_count": prior_fires.size().getInfo(),
        "prior_overlap_pct": prior_overlap_pct.getInfo(),
        "prior_names": prior_fires.aggregate_array("Incid_Name").getInfo(),
        "reburn_count": reburn_fires.size().getInfo(),
        "reburn_overlap_pct": reburn_overlap_pct.getInfo(),
        "reburn_names": reburn_fires.aggregate_array("Incid_Name").getInfo(),
    }


candidate_ids = [
    "CA3765211988120180713",  # Ferguson
    "CA4178612347520170812",  # Oak
    "ID4549011522720170729",  # Highline
    "OR4509112124520170809",  # Nena Springs
    "UT4010011096020180701",  # Dollar Ridge
    "MT4599511358020170714",  # Meyers
]


results = []

for event_id in candidate_ids:
    results.append(screen_fire_history(event_id))


history_df = pd.DataFrame(results)

history_df[
    [
        "name",
        "year",
        "acres",
        "prior_fire_count",
        "prior_overlap_pct",
        "reburn_count",
        "reburn_overlap_pct",
    ]
].round(2)

,name,year,acres,prior_fire_count,prior_overlap_pct,reburn_count,reburn_overlap_pct
0,FERGUSON,2018,97307,4,4.00,2,4.38
1,OAK,2017,95692,1,0.14,2,1.98
2,HIGHLINE,2017,93699,3,1.07,2,0.15
3,NENA SPRINGS,2017,70074,6,63.41,0,0.00
4,DOLLAR RIDGE,2018,69817,0,0.00,0,0.00
5,MEYERS,2017,68711,2,3.31,1,0.32


In [15]:
#Does the fire actually contain a substantial amount of tree-covered landscape?
#this will NOT become our final forest/woodland eligibility mask. proper pre-fire LANDFIRE vegetation/ecosystem classification, likely EVT will be done soon, just using EVC as a quick candidate-selection proxy.
evc_collection = ee.ImageCollection(
    "LANDFIRE/Vegetation/EVC/v1_4_0"
)

print("Number of EVC images:", evc_collection.size().getInfo())

print(
    "Image IDs:",
    evc_collection.aggregate_array("system:index").getInfo()
)

Number of EVC images: 3
Image IDs: ['AK', 'CONUS', 'HI']


In [16]:
# filter to conus
evc_2016 = (
    evc_collection
    .filter(ee.Filter.eq("system:index", "CONUS"))
    .first()
    .select("EVC")
)
# LANDFIRE EVC tree-cover classes:
# 101–109 = tree cover from 10% through 100%
# 151–153 = broader tree-canopy classes
tree_mask = (
    evc_2016.gte(101).And(evc_2016.lte(109))
    .Or(
        evc_2016.gte(151).And(evc_2016.lte(153))
    )
    .rename("tree")
)


In [17]:
# percent of fire perimeter that is tree covered
finalist_ids = [
    "CA3765211988120180713",  # Ferguson
    "CA4178612347520170812",  # Oak
    "ID4549011522720170729",  # Highline
    "UT4010011096020180701",  # Dollar Ridge
    "MT4599511358020170714",  # Meyers
]


def tree_cover_pct(event_id):
    focal = ee.Feature(
        western_wildfires_2013_2020
        .filter(ee.Filter.eq("Event_ID", event_id))
        .first()
    )

    stats = tree_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=focal.geometry(),
        scale=30,
        maxPixels=1e9
    )

    return {
        "name": focal.get("Incid_Name").getInfo(),
        "tree_pct": stats.get("tree").getInfo() * 100
    }


tree_results = []

for event_id in finalist_ids:
    tree_results.append(tree_cover_pct(event_id))


tree_df = pd.DataFrame(tree_results)

tree_df.round(2)

,name,tree_pct
0,FERGUSON,90.23
1,OAK,83.78
2,HIGHLINE,95.48
3,DOLLAR RIDGE,71.04
4,MEYERS,94.45


### Provisionally choosing HIGHLINE for its clean history + heterogeneous real forest ecology

#### two more things to check before locking in HIGHLINE as POC candidate
#### 1) Does MTBS show useful burn-severity variation inside Highline?
#### 2) Can we retrieve/visualize its perimeter and severity cleanly?

In [18]:
# retrieving highline clearly
highline_id = "ID4549011522720170729"

highline = ee.Feature(
    western_wildfires_2013_2020
    .filter(ee.Filter.eq("Event_ID", highline_id))
    .first()
)

highline_info = highline.toDictionary([
    "Event_ID",
    "Incid_Name",
    "Incid_Type",
    "Ig_Date",
    "BurnBndAc",
    "BurnBndLat",
    "BurnBndLon",
    "Low_T",
    "Mod_T",
    "High_T"
]).getInfo()

highline_info

{'BurnBndAc': 93699,
 'BurnBndLat': '45.364',
 'BurnBndLon': '-115.156',
 'Event_ID': 'ID4549011522720170729',
 'High_T': 470,
 'Ig_Date': 1501311600000,
 'Incid_Name': 'HIGHLINE',
 'Incid_Type': 'Wildfire',
 'Low_T': 50,
 'Mod_T': 243}

#### inspecting mtbs burn severity raster for highline

In [19]:
# Load the 2017 entirety of CONUS MTBS burn-severity raster and select its Severity band.
# severity is already calculated before it goes into the raster based on thresholds set for different fires (we dont need to do it)

# we are reading already-classified severity pixels; we are not applying Highline's 50/243/470 thresholds ourselves.
mtbs_severity = ee.ImageCollection(
    "USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1"
)

severity_2017_matches = mtbs_severity.filter(
    ee.Filter.eq("system:index", "mtbs_CONUS_2017")
)

print(
    "Matching 2017 CONUS images:",
    severity_2017_matches.size().getInfo()
)

severity_2017 = (
    severity_2017_matches
    .first()
    .select("Severity")
)

print(
    "Bands:",
    severity_2017.bandNames().getInfo()
)

Matching 2017 CONUS images: 1
Bands: ['Severity']


In [20]:
#Count how many 30 m pixels of each severity class fall inside the Highline fire perimeter.
highline_severity_hist = severity_2017.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=highline.geometry(),
    scale=30,
    maxPixels=1e9
)

highline_severity_hist.getInfo()

{'Severity': {'1': 58271.9176470587,
  '2': 131043.87450980392,
  '3': 134214.0274509803,
  '4': 94661.19607843139,
  '5': 1056.6117647058823}}

Class 1 — Unburned to low: ~13.9%
Class 2 — Low: ~31.3%
Class 3 — Moderate: ~32.0%
Class 4 — High: ~22.6%
Class 5 — Increased greenness: ~0.25%

HIGHLAND has passed all current screens:

~93,700 acres — useful size  
2017 — ideal timing for Landsat pre/Season1/Season5  
~95.5% tree-covered by quick LANDFIRE proxy  
only ~1.1% recent prior-fire overlap  
only ~0.15% reburn overlap through Season 5  
strong mix of low/moderate/high burn severity  


#### Choosing HIGHLINE as Phase 2 focal POC fire, unless the Landsat test tomorrow reveals some unexpected data-quality issue.

# visual sanity check of Highline’s perimeter and severity map

In [21]:
# Create a map centered on Highline
import geemap

Map = geemap.Map()
Map.centerObject(highline, 9)

Map

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

In [22]:
# adding two data layers

# 1) the 2017 MTBS severity raster inside Highline
# 2) Highline’s fire perimeter outline

severity_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "7fff00",  # 1 = Unburned to Low
        "ffff00",  # 2 = Low
        "ff9900",  # 3 = Moderate
        "ff0000",  # 4 = High
        "00bfff",  # 5 = Increased Greenness
    ],
}

Map.addLayer(
    severity_2017.clip(highline.geometry()),
    severity_vis,
    "Highline MTBS Severity"
)

highline_outline = ee.FeatureCollection([highline]).style(
    color="000000",
    fillColor="00000000",
    width=3
)

Map.addLayer(
    highline_outline,
    {},
    "Highline Perimeter"
)

Map.centerObject(highline, 10)

Map

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

### Save important views as images instead of just relying on the map

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

# Resolve repository root
cwd = Path.cwd()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

output_dir = repo_root / "outputs" / "figures" / "mtbs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "highline_mtbs_severity.png"


# Use the EXACT same MTBS visualization as the notebook map
severity_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "7fff00",  # 1 = Unburned to Low
        "ffff00",  # 2 = Low
        "ff9900",  # 3 = Moderate
        "ff0000",  # 4 = High
        "00bfff",  # 5 = Increased Greenness
    ],
}

severity_rgb = (
    severity_2017
    .clip(highline.geometry())
    .visualize(**severity_vis)
)

# Black Highline perimeter
outline = (
    ee.Image()
    .byte()
    .paint(ee.FeatureCollection([ee.Feature(highline)]), 1, 3)
    .selfMask()
    .visualize(palette=["000000"])
)

# Overlay perimeter on severity image
figure_image = severity_rgb.blend(outline)

# Request clean PNG from Earth Engine
thumb_url = figure_image.getThumbURL({
    "region": highline.geometry(),
    "dimensions": 1600,
    "format": "png",
})

urlretrieve(thumb_url, output_path)

print("Saved:")
print(output_path)

Saved:
c:\F copy\wildfire-recovery-forecasting\outputs\figures\mtbs\highline_mtbs_severity.png


# SUMMARY
30,730 MTBS events → 3,644 wildfires in 2013–20 → 2,282 western fires → candidate screening → Highline → fire-history check → forest-context check → severity distribution → actual map.

## Phase 2 POC Fire Selection

Highline (Idaho, 2017; MTBS Event ID `ID4549011522720170729`) was selected as the provisional proof-of-concept wildfire.

Reasons:
- 93,699-acre MTBS wildfire, providing a sufficiently large test area.
- ~95.5% tree-covered using LANDFIRE EVC as a preliminary forest-context proxy.
- Only ~1.1% of the perimeter overlaps wildfires from the preceding 7 years.
- Only ~0.15% overlaps subsequent wildfires through the end of Season 5.
- Contains substantial low, moderate, and high MTBS burn-severity areas.
- 2017 timing provides convenient Landsat 8 coverage for pre-fire, Season 1, and Season 5 testing.

These screening choices are for Phase 2 proof-of-concept convenience only and do not define final study-population exclusion thresholds.